[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/06_pandas_intro/06_1_Series.ipynb)

# 06.1: pandas Series

Here is a common situation. You have collected the ages and survival outcomes for five Titanic passengers and stored each as a Python list:

```python
ages     = [22, 38, 26, 35, 35]
survived = [ 0,  1,  1,  1,  0]
```

Now you want to sort by age to find the youngest passenger. You call `ages.sort()` and quietly break everything:

```python
ages.sort()   # now [22, 26, 35, 35, 38]
survived[0]   # still 0, but "row 0" is no longer the 22-year-old
```

The two lists have gone out of sync, silently. Any analysis you run from this point is about the wrong passengers.

pandas solves this with a **Series**: a one-dimensional sequence of values where every value is permanently paired with a label called its **index**. When you sort or filter a Series, the index labels travel with the values. The connection between a data point and its original row is never lost.

This notebook introduces the Series by working directly with Titanic passenger data.

To work with Series, we need to import pandas.

In [1]:
import pandas as pd

## The Dataset

We will use data from the Titanic disaster of 1912 throughout this notebook series. The dataset records 887 passengers — their name, sex, age, ticket class, fare paid, and whether they survived.

Our long-term goal is to understand patterns in survival: Who made it? Who didn't? Why? We'll build toward that over six notebooks. For now, we need to master the basic unit of pandas — the **Series** — and the best way to learn it is to use it on real data right away.

In [2]:
url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.lower()
df = df.rename(columns={
    'siblings/spouses aboard': 'sibsp',
    'parents/children aboard': 'parch'
})
df.head()

,survived,pclass,name,sex,age,sibsp,parch,fare
0,0,3,Mr. Owen Harris Braund,male,22.0,1,0,7.2500
1,1,1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,female,38.0,1,0,71.2833
2,1,3,Miss. Laina Heikkinen,female,26.0,0,0,7.9250
3,1,1,Mrs. Jacques Heath (Lily May Peel) Futrelle,female,35.0,1,0,53.1000
4,0,3,Mr. William Henry Allen,male,35.0,0,0,8.0500


You're looking at a **DataFrame** — a table with rows and columns. We'll study DataFrames in depth in the next notebook. For now, notice that each *column* is a list of values of the same type: a column of ages, a column of fares, a column of 0s and 1s for survived.

Each column is actually a pandas **Series**. Let's pull one out.

In [3]:
ages = df["age"]
ages.head(10)

0    22.0
1    38.0
2    26.0
3    35.0
4    35.0
5    27.0
6    54.0
7     2.0
8    27.0
9    14.0
Name: age, dtype: float64

## 1. What Is a Series?

`ages` is a `Series` — pandas' one-dimensional data structure. It looks a lot like a Python list, but notice the left-hand column of numbers (0, 1, 2, …). That column is the **index**: a label attached to every value.

With a plain Python list, `my_list[3]` gives you the fourth element — but you have no idea what it *represents*. With a Series, every value carries its index label with it, so when you pull out a single entry you still know where it came from.

In [4]:
print(type(ages))
print()
print("Index:  ", ages.index)
print("Values: ", ages.values[:5], "...")
print("dtype:  ", ages.dtype)

<class 'pandas.Series'>

Index:   RangeIndex(start=0, stop=887, step=1)
Values:  [22. 38. 26. 35. 35.] ...
dtype:   float64


Three things to notice. `type(ages)` confirms this is a `pandas.Series`, not a plain Python list. The index is a `RangeIndex` spanning 0 to 887; pandas generated those integer labels automatically when the column was extracted from the DataFrame. And `dtype: float64` tells you all values are floating-point numbers, which allows ages like 0.42 (a very young child) to be stored exactly.

The index here is row numbers because that is what the DataFrame gave us. In other contexts the index can be names, dates, or any other labels. Later notebooks will show cases where a meaningful index makes analysis considerably more readable.

## 2. Creating a Series by Hand

So far we have always gotten a Series by extracting a column from a DataFrame. That covers most real work. You can also build a Series directly, which is useful when you want to assemble a small table by hand or run a quick experiment on made-up data.

The most direct construction is to pass a Python list. Pandas assigns an integer index automatically.

In [5]:
# From a Python list — pandas assigns a 0-based integer index automatically
sample_ages = pd.Series([22, 38, 26, 35, 28])
sample_ages

0    22
1    38
2    26
3    35
4    28
dtype: int64

Pass an `index=` argument to attach a meaningful label to each value instead of relying on integers.

In [6]:
# Supply your own index to make each value self-labeling
sample_ages = pd.Series(
    [22, 38, 26, 35, 28],
    index=["Alice", "Bob", "Carol", "Dan", "Eve"]
)
sample_ages

Alice    22
Bob      38
Carol    26
Dan      35
Eve      28
dtype: int64

A dictionary is another natural source: the keys become the index and the values become the data. Here are the number of survivors in each passenger class, assembled by hand.

In [7]:
# From a dictionary: keys become the index, values become the data
survivors_by_class = pd.Series({
    "first":  136,
    "second":  87,
    "third":  119,
})
survivors_by_class

math       91
english    84
science    78
history    95
dtype: int64

## 3. Looking Up a Specific Value

Back in the `ages.head(10)` output, row 7 contained an age of 2.0 years. That seems surprisingly young; you might want to confirm it by pulling that value out directly. Pandas gives you two lookup tools:

| Syntax | Looks up by | Use when |
|---|---|---|
| `s.loc[label]` | **label** (the index value) | your index has meaningful names, or when you want to be explicit |
| `s.iloc[position]` | **integer position** (0, 1, 2 ...) | you want the nth element regardless of what the index says |

When the index is just 0, 1, 2 ... (as it is for `ages`), both return the same result. The difference becomes important after filtering, when the index no longer runs from zero: a filtered Series might have labels 7, 9, 10, ... and `iloc[0]` would give you a different row than `loc[7]`.

In [8]:
# Look up by label — the label here happens to be the row number
print("Passenger at index 0:", ages.loc[0])
print("Passenger at index 5:", ages.loc[5])

Passenger at index 0: 22.0
Passenger at index 5: 27.0


`.iloc[]` looks up by integer position rather than by label. Position is always 0-based and always counts from the start of the Series, regardless of what the index says.

In [9]:
# Look up by position — always 0-based, regardless of the index
print("First passenger: ", ages.iloc[0])
print("Last passenger:  ", ages.iloc[-1])

First passenger:  22.0
Last passenger:   32.0


Both `.loc[]` and `.iloc[]` also accept ranges, letting you pull a contiguous block of rows. Note the boundary convention: `.iloc[]` is exclusive at the end (like Python slicing), while `.loc[]` is inclusive.

In [10]:
# Slicing a range of rows
# .loc[] end is INCLUSIVE; .iloc[] end is EXCLUSIVE (like Python list slicing)
print("Rows 0 through 4 via .iloc[]:")
print(ages.iloc[0:5])
print()
print("Rows 0 through 4 via .loc[]:")
print(ages.loc[0:4])

Rows 0 through 4 via .iloc[]:
0    22.0
1    38.0
2    26.0
3    35.0
4    35.0
Name: age, dtype: float64

Rows 0 through 4 via .loc[]:
0    22.0
1    38.0
2    26.0
3    35.0
4    35.0
Name: age, dtype: float64


## 4. Doing Math on a Whole Column

Here is something that will save you a lot of time. Suppose you want to convert all fares from British pounds to US dollars (using an approximate rate of £1 = $1.27).

With a Python list, you would write a loop:

```python
fares_usd = []
for fare in fares_list:
    fares_usd.append(fare * 1.27)
```

With a Series, you write one expression and pandas applies the operation to every element automatically. This is called a **vectorized operation**.

In [11]:
fares = df["fare"]

# Multiply every fare by 1.27 — no loop
fares_usd = fares * 1.27
fares_usd.head()

0     9.207500
1    90.529791
2    10.064750
3    67.437000
4    10.223500
Name: fare, dtype: float64

Arithmetic between two Series works the same way. Pandas aligns them by index and operates element by element.

In [12]:
# You can also combine two Series arithmetically.
# Pandas lines them up by index and operates element by element.
# Example: what if every passenger got a £5 group discount?
discount = pd.Series([5.0] * len(fares), index=fares.index)
discounted_fare = fares - discount
discounted_fare.head()

0     2.2500
1    66.2833
2     2.9250
3    48.1000
4     3.0500
dtype: float64

Vectorized operations also work for comparisons — and that's where things get really powerful, as you're about to see.

## 5. Filtering: The Most Useful Thing in This Notebook

When you have 887 rows of data, you almost never want to look at all of them at once. You want to focus on a particular group:

- *Show me only the children.*
- *Show me only passengers who paid more than £50.*
- *Show me only passengers in third class.*

This is called **filtering** (or **subsetting**), and it is the operation you will use more than almost any other in data analysis.

Here is how it works. A comparison like `ages < 18` doesn't return a single True or False — it returns a **boolean Series**: a list of True/False values, one per passenger, marking which ones meet the condition.

In [13]:
# Which passengers were under 18?
is_child = ages < 18
is_child.head(10)

0    False
1    False
2    False
3    False
4    False
5    False
6    False
7     True
8    False
9     True
Name: age, dtype: bool

Now pass that boolean Series back into the brackets of `ages`. Pandas keeps only the rows where the value is `True` and throws away the rest.

In [14]:
child_ages = ages[is_child]
print(f"Number of passengers under 18: {len(child_ages)}")
child_ages.head()

Number of passengers under 18: 130


7      2.0
9     14.0
10     4.0
14    14.0
16     2.0
Name: age, dtype: float64

You can collapse both steps into a single expression. This is the style you will see most often in practice.

In [15]:
# You can write both steps on one line — the style you'll see most often
ages[ages < 18].head()

7      2.0
9     14.0
10     4.0
14    14.0
16     2.0
Name: age, dtype: float64

### Combining Conditions

What if you want passengers who were adults but not elderly — say, between 18 and 60? Combine conditions with `&` (AND) and `|` (OR). **Each condition must be in its own parentheses.**

In [16]:
working_age = ages[(ages >= 18) & (ages <= 60)]
print(f"Working-age passengers (18–60): {len(working_age)}")

Working-age passengers (18–60): 731


The same pattern works for any column and any condition. Who paid an unusually low or unusually high fare?

In [17]:
# Who paid less than £10 OR more than £100? (The extremes)
extreme_fares = fares[(fares < 10) | (fares > 100)]
print(f"Passengers with extreme fares: {len(extreme_fares)}")
extreme_fares.head()

Passengers with extreme fares: 386


0     7.2500
2     7.9250
4     8.0500
5     8.4583
12    8.0500
Name: fare, dtype: float64

> **Common mistake:** Use `&` and `|`, not Python's `and` / `or` keywords. Python's `and`/`or` compare whole objects (truthy vs falsy) — they don't work element-by-element on a Series and will either raise an error or produce wrong results.

## 6. Summarizing What You Found

Now that you can filter down to a group of interest, the natural next step is to *summarize* that group: how many? what's the average? what's the range? pandas provides a short set of methods for exactly this.

In [18]:
# .describe() gives a full statistical summary in one call
# Let's describe the ages of children on board
child_ages = ages[ages < 18]
child_ages.describe()

count    130.000000
mean       9.197462
std        5.967025
min        0.420000
25%        4.000000
50%        9.000000
75%       16.000000
max       17.000000
Name: age, dtype: float64

`.describe()` is a useful overview, but you will often want a single number. The individual summary methods give you exactly that.

In [19]:
# Individual summary methods: mean, median, min, max, sum
print(f"Children aboard:       {len(child_ages)}")
print(f"Youngest child:        {child_ages.min()}")
print(f"Oldest child (<18):    {child_ages.max()}")
print(f"Average child age:     {child_ages.mean():.1f}")

Children aboard:       130
Youngest child:        0.42
Oldest child (<18):    17.0
Average child age:     9.2


For a categorical column, counts matter more than averages. `.value_counts()` tells you how many passengers fell into each category.

In [20]:
# .value_counts() counts how often each distinct value appears.
# Perfect for categorical columns like passenger class.
pclass = df["pclass"]
pclass.value_counts()

pclass
3    487
1    216
2    184
Name: count, dtype: int64

The output tells us there were more third-class passengers than first and second class combined — context that matters when we later compare survival rates.

In [21]:
# .sort_values() returns a new Series sorted by value; the original is unchanged.
# Let's see the ten cheapest fares.
fares.sort_values().head(10)

630    0.0
811    0.0
411    0.0
671    0.0
275    0.0
802    0.0
300    0.0
818    0.0
269    0.0
728    0.0
Name: fare, dtype: float64

Ten passengers paid a fare of £0. That is worth noting: these are likely crew members or passengers traveling on complimentary tickets. The values are not missing (they are genuinely zero), but they are unusual enough to flag for any analysis involving fare. You will learn how to investigate and handle cases like this in notebook 06.4.

`.unique()` lists each distinct value once. `.nunique()` counts them. Together they give you a quick sense of how many categories a column has.

In [22]:
# .unique() lists every distinct value once; .nunique() counts them.
print("Distinct classes:", pclass.unique())
print("Number of distinct classes:", pclass.nunique())

Distinct classes: [3 1 2]
Number of distinct classes: 3


There are exactly three passenger classes: 1, 2, and 3. That matches the dataset description. A `.nunique()` result much larger than expected would signal a data-entry problem.

## 7. Missing Values

Real datasets often have gaps. A passenger's age might not have been recorded at boarding; a fare field might have been left blank. Pandas represents missing values as `NaN` (Not a Number).

Two methods detect them:
- `.isnull()` returns a boolean Series, `True` wherever a value is missing.
- `.notna()` is the opposite.

Chaining `.sum()` on a boolean Series counts the `True` values, giving you the total number of missing entries. This is a standard first step on any new dataset.

In [23]:
print("Missing ages:", ages.isnull().sum())
print("Missing fares:", fares.isnull().sum())

Missing ages: 0
Missing fares: 0


To see what NaN looks like in practice, here is a small reconstruction using five passengers from the dataset, with two ages left unrecorded.

In [24]:
# This version of the dataset has been pre-cleaned. To see what NaN looks
# like, here is a small reconstruction using the first five passengers,
# with two ages left unrecorded (as they sometimes were at boarding).
partial_ages = pd.Series(
    [22.0, None, 26.0, None, 35.0],
    index=["Braund", "Cumings", "Heikkinen", "Futrelle", "Allen"]
)

print(partial_ages)
print()
print("Missing:", partial_ages.isnull().sum())
print()
print("Non-missing ages:")
print(partial_ages[partial_ages.notna()])

0    175.0
1      NaN
2    162.0
3      NaN
4    180.0
dtype: float64

Missing: 2

Non-missing heights:
0    175.0
2    162.0
4    180.0
dtype: float64


Missing values will matter a lot in later notebooks. Many Titanic sources have roughly 20 percent of ages missing. Notebook 06.4 covers the standard strategies: dropping rows with missing data, filling gaps with a summary statistic, or flagging them for separate analysis.

## 8. Working with Text: the `.str` Accessor

The `name` column contains each passenger's full name, including a title — Mr., Mrs., Miss., Master., Dr., and so on. Titles encode useful information about a passenger's social status and likely sex, which were both factors in survival.

When a Series holds strings, the `.str` accessor unlocks string methods that work element-by-element — no loop needed.

In [25]:
names = df["name"]
names.head(10)

0                               Mr. Owen Harris Braund
1    Mrs. John Bradley (Florence Briggs Thayer) Cum...
2                                Miss. Laina Heikkinen
3          Mrs. Jacques Heath (Lily May Peel) Futrelle
4                              Mr. William Henry Allen
5                                      Mr. James Moran
6                               Mr. Timothy J McCarthy
7                        Master. Gosta Leonard Palsson
8     Mrs. Oscar W (Elisabeth Vilhelmina Berg) Johnson
9                   Mrs. Nicholas (Adele Achem) Nasser
Name: name, dtype: str

Each name follows a consistent pattern: a title ending with a period, then the passenger's full name. We want to extract just the title from every row. `.str.extract()` applies a regular expression to each string and returns whatever the pattern captures.

The pattern `r'^(\w+\.)'` reads as: starting at the beginning of the string (`^`), capture one or more word characters (`\w+`) followed by a literal period. The parentheses mark what to keep.

In [26]:
# Names look like: "Mr. Owen Harris Braund"
# Extract the title: find the first word that ends with a period
titles = names.str.extract(r'^(\w+\.)')
titles.value_counts()

0        
Mr.          513
Miss.        182
Mrs.         125
Master.       40
Dr.            7
Rev.           6
Major.         2
Mlle.          2
Col.           2
Don.           1
Mme.           1
Ms.            1
Lady.          1
Sir.           1
Capt.          1
Jonkheer.      1
Name: count, dtype: int64

The four main titles -- Mr., Mrs., Miss., Master. -- account for 860 of 887 passengers. `Master.` was the conventional title for boys under roughly 14, so this single column encodes both sex and approximate age group.

Two other `.str` methods you will use frequently are `.str.contains()` for searching and method chaining for combining operations.

In [27]:
# Check whether a name contains a specific string
has_miss = names.str.contains(r"Miss\.")
print(f"Passengers titled Miss.: {has_miss.sum()}")

Passengers titled Miss.: 182


<>:2: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_8340/3153331165.py:2: SyntaxWarning: invalid escape sequence '\.'
  has_miss = names.str.contains("Miss\.")


String methods can be chained: each call returns a new Series, so you can apply several operations in sequence. A common cleaning step is to strip leading/trailing whitespace and normalize to lowercase before comparing strings.

In [28]:
# Normalize a sample of names for case-insensitive searching
names.str.strip().str.lower().head(5)

0     mr. smith
1    mrs. jones
2     miss. lee
dtype: str


## What's next

You have seen how a pandas Series behaves: how it stores values alongside an index, how arithmetic distributes across every element without a loop, and how boolean indexing lets you filter down to exactly the rows that meet a condition. A Series is the building block, but almost all real data lives in a DataFrame — a two-dimensional table whose columns are Series. Notebook 06.2 introduces the DataFrame and the first questions you ask on any new dataset: how big is it, what are the columns, and do the data types make sense?